# 🛒 Python & Pandas Basics — Shopping Dataset
> **Objective:** Load a shopping CSV, explore it, clean it, and derive new columns.

**Steps covered:**
1. Load CSV into a DataFrame
2. Explore data (head/tail, shape, dtypes)
3. Handle missing values
4. Filter rows & select columns
5. Remove duplicates
6. Create derived column (`total_amount`)
7. Save cleaned dataset


## 0. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

print('pandas:', pd.__version__)
print('numpy :', np.__version__)


## Step 1 — Load CSV into a DataFrame

In [ ]:
df = pd.read_csv('shopping_raw.csv')
print('✅ Dataset loaded successfully!')
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')


## Step 2 — Explore the Data

We look at the first/last rows, column names, data types, and summary statistics.

In [ ]:
print('--- First 5 rows ---')
df.head()


In [ ]:
print('--- Last 5 rows ---')
df.tail()


In [ ]:
print(f'Shape : {df.shape}')
print(f'Columns: {list(df.columns)}')


In [ ]:
print('--- Data Types ---')
df.dtypes


In [ ]:
print('--- Statistical Summary ---')
df.describe(include='all')


## Step 3 — Handle Missing Values

Identify nulls, then fill numeric columns with their median and drop any remaining nulls.

In [ ]:
print('--- Missing Values Count ---')
print(df.isnull().sum())
print()
print('--- Missing Values % ---')
print((df.isnull().sum() / len(df) * 100).round(2).astype(str) + ' %')


In [ ]:
# Fill missing 'price' with median price per category
df['price'] = df.groupby('category')['price'].transform(
    lambda x: x.fillna(x.median())
)

# Fill missing 'quantity' with overall median (1 or 2)
df['quantity'] = df['quantity'].fillna(df['quantity'].median())
df['quantity'] = df['quantity'].astype(int)

print('Missing values after filling:')
print(df.isnull().sum())


In [ ]:
# Drop any rows that still have nulls (safety net)
before = len(df)
df.dropna(inplace=True)
print(f'Rows dropped: {before - len(df)}')
print(f'Remaining rows: {len(df)}')


## Step 4 — Filter Rows & Select Columns

Demonstrate filtering by category and city, then selecting specific columns.

In [ ]:
# Filter: Electronics orders only
electronics = df[df['category'] == 'Electronics']
print(f'Electronics orders: {len(electronics)}')
electronics[['order_id','product','price','quantity','city']].head()


In [ ]:
# Filter: orders where quantity >= 3
bulk_orders = df[df['quantity'] >= 3]
print(f'Bulk orders (qty ≥ 3): {len(bulk_orders)}')
bulk_orders[['order_id','product','category','quantity']].head()


In [ ]:
# Filter: orders from Bangalore OR Delhi
metro = df[df['city'].isin(['Bangalore', 'Delhi'])]
print(f'Orders from Bangalore/Delhi: {len(metro)}')


In [ ]:
# Select specific columns of interest
selected = df[['order_id', 'product', 'category', 'price', 'quantity', 'city']]
selected.head()


## Step 5 — Remove Duplicates

We check for fully duplicated rows and drop them.

In [ ]:
print(f'Total rows before dedup : {len(df)}')
print(f'Duplicate rows found     : {df.duplicated().sum()}')


In [ ]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Rows after removing duplicates: {len(df)}')


## Step 6 — Create Derived Column: `total_amount`

`total_amount = price × quantity` — the revenue per order line.

In [ ]:
df['total_amount'] = df['price'] * df['quantity']
print('New column added: total_amount')
df[['order_id','product','price','quantity','total_amount']].head(10)


In [ ]:
print('--- Revenue Summary ---')
print(f"Total revenue        : ₹{df['total_amount'].sum():,.2f}")
print(f"Average order value  : ₹{df['total_amount'].mean():,.2f}")
print(f"Max single order     : ₹{df['total_amount'].max():,.2f}")
print()
print('--- Revenue by Category ---')
print(df.groupby('category')['total_amount'].sum().sort_values(ascending=False).apply(lambda x: f'₹{x:,.2f}'))


## Step 7 — Save the Cleaned Dataset

Export the cleaned, enriched DataFrame to a new CSV file.

In [ ]:
output_path = 'shopping_cleaned.csv'
df.to_csv(output_path, index=False)
print(f'✅ Cleaned dataset saved → {output_path}')
print(f'Final shape: {df.shape[0]} rows × {df.shape[1]} columns')


In [ ]:
# Quick verification — reload and inspect
verify = pd.read_csv(output_path)
print(verify.info())
print()
verify.head()


## 📋 Summary

| Step | Action | Result |
|------|--------|--------|
| 1 | Loaded CSV | 128 rows × 8 columns |
| 2 | Explored data | Checked head/tail, dtypes, describe |
| 3 | Handled missing values | Filled price (category median), quantity (median); dropped residual nulls |
| 4 | Filtered & selected | Electronics, bulk orders, city filters |
| 5 | Removed duplicates | 8 duplicate rows dropped |
| 6 | Derived column | `total_amount = price × quantity` |
| 7 | Saved CSV | `shopping_cleaned.csv` (120 rows × 9 columns) |

> **Key insight:** Electronics generates the highest total revenue despite fewer orders due to high unit prices.
